In [106]:
import pandas as pd
import os

In [107]:
df_rent = pd.read_csv('../data/cian_commercial_150_700.csv')
df_rent.drop(columns=['id'], inplace=True)
df_rent.head()

,address,area_m2,url
0,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...
1,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",340.8,https://www.cian.ru/rent/commercial/306607073/...
2,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",629.8,https://www.cian.ru/rent/commercial/306607073/...
3,"Москва, ТАО (Троицкий), м. Апрелевка, д. Сеньк...",563.0,https://www.cian.ru/rent/commercial/323315566/...
4,"Москва, ВАО, р-н Гольяново, м. Черкизовская, Щ...",252.2,https://www.cian.ru/rent/commercial/326542571/...


In [108]:
df_rent['Регион'] = df_rent['address'].apply(lambda d: d.split(', ')[0] if d.split(', ')[0] not in ['Москва', 'Санкт-Петербург'] else f'{d.split(', ')[0]} г')
df_rent['Регион'] = df_rent['Регион'].apply(lambda d: f'{d.split()[0]} обл' if d.split()[1] == 'область' else d)
df_rent['Регион'] = df_rent['Регион'].apply(lambda d: f'{d.split()[1]} Респ' if d == 'Республика Татарстан' else d)
df_rent['Регион'].unique()

<StringArray>
[          'Москва г',  'Санкт-Петербург г',     'Татарстан Респ',
     'Ростовская обл',   'Свердловская обл',  'Новосибирская обл',
 'Краснодарский край',      'Самарская обл']
Length: 8, dtype: str

In [109]:
df_rent['Населенный пункт'] = df_rent['address'].apply(lambda d: f'{d.split(', ')[1]} г' if d.split(', ')[0] not in ['Москва', 'Санкт-Петербург'] else f'{d.split(', ')[0]} г')
df_rent['Населенный пункт'].unique()

<StringArray>
[         'Москва г', 'Санкт-Петербург г',          'Казань г',
  'Ростов-на-Дону г',    'Екатеринбург г',     'Новосибирск г',
       'Краснодар г',          'Самара г']
Length: 8, dtype: str

In [110]:
df_rent.rename(columns={
    'address': 'Полный адрес помещения',
    'area_m2': 'Торговая площадь, вещественный'
}, inplace=True)
df_rent.head()

,Полный адрес помещения,"Торговая площадь, вещественный",url,Регион,Населенный пункт
0,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г
1,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",340.8,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г
2,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",629.8,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г
3,"Москва, ТАО (Троицкий), м. Апрелевка, д. Сеньк...",563.0,https://www.cian.ru/rent/commercial/323315566/...,Москва г,Москва г
4,"Москва, ВАО, р-н Гольяново, м. Черкизовская, Щ...",252.2,https://www.cian.ru/rent/commercial/326542571/...,Москва г,Москва г


In [111]:
mask_small = (df_rent['Торговая площадь, вещественный'] > 150) & (df_rent['Торговая площадь, вещественный'] < 300)
mask_middle = (df_rent['Торговая площадь, вещественный'] > 300) & (df_rent['Торговая площадь, вещественный'] < 550)
mask_big = df_rent['Торговая площадь, вещественный'] > 550

df_rent['Торговая площадь, категориальный'] = ''
df_rent.loc[mask_small, 'Торговая площадь, категориальный'] = 'Маленький'
df_rent.loc[mask_middle, 'Торговая площадь, категориальный'] = 'Средний'
df_rent.loc[mask_big, 'Торговая площадь, категориальный'] = 'Большой'

df_rent.head()

,Полный адрес помещения,"Торговая площадь, вещественный",url,Регион,Населенный пункт,"Торговая площадь, категориальный"
0,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Маленький
1,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",340.8,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Средний
2,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",629.8,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Большой
3,"Москва, ТАО (Троицкий), м. Апрелевка, д. Сеньк...",563.0,https://www.cian.ru/rent/commercial/323315566/...,Москва г,Москва г,Большой
4,"Москва, ВАО, р-н Гольяново, м. Черкизовская, Щ...",252.2,https://www.cian.ru/rent/commercial/326542571/...,Москва г,Москва г,Маленький


In [112]:
df_radius_stats = pd.read_csv('../data/addresses_w_poi.csv')
df_radius_stats.drop(columns=['address'], inplace=True)

df_radius_stats.rename(columns={
    'маркетплейсы, постаматы (100м)': 'Маркетплейсы, доставки, постаматы (100 м)',
    'медицина и аптеки (300м)': 'Медицинские уч. и аптеки (300 м)',
    'школы (300м)': 'Школы (300 м)',
    'остановки (300м)': 'Остановки (300 м)',
    'продуктовые магазины (500м)': 'Продуктовые магазины (500 м)',
    'пятерочки (500м)': 'Пятерочки (500 м)'
}, inplace=True)

df_radius_stats.head()

,"Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м)
0,0,0,2,2,4,0
1,0,0,2,2,4,0
2,0,0,2,2,4,0
3,0,0,0,1,0,0
4,0,5,0,2,8,1


In [113]:
print(df_radius_stats.shape, df_rent.shape)

(328, 6) (329, 6)


In [114]:
df_radius_stats = pd.concat([
    df_radius_stats.iloc[:256, :],
    df_radius_stats.iloc[[256]], # иначе если df_radius_stats.iloc[256], это pd.Series, а не pd.DataFrame - получится неверная конкатенация, добавится столбец с названием "256"
    df_radius_stats.iloc[256:, :]
]).reset_index(drop=True) # из-за того, что строку вставили индексация полетела, есть дублирующиеся индексы. Для этого сбрасываем их, чтобы все строки заново проиндексировались
df_radius_stats.shape

(329, 6)

In [115]:
print(df_radius_stats.shape, df_rent.shape)

(329, 6) (329, 6)


In [116]:
df = pd.concat([df_rent, df_radius_stats], axis=1)
df.head()

,Полный адрес помещения,"Торговая площадь, вещественный",url,Регион,Населенный пункт,"Торговая площадь, категориальный","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м)
0,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Маленький,0,0,2,2,4,0
1,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",340.8,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Средний,0,0,2,2,4,0
2,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",629.8,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Большой,0,0,2,2,4,0
3,"Москва, ТАО (Троицкий), м. Апрелевка, д. Сеньк...",563.0,https://www.cian.ru/rent/commercial/323315566/...,Москва г,Москва г,Большой,0,0,0,1,0,0
4,"Москва, ВАО, р-н Гольяново, м. Черкизовская, Щ...",252.2,https://www.cian.ru/rent/commercial/326542571/...,Москва г,Москва г,Маленький,0,5,0,2,8,1


In [117]:
df.duplicated(subset=[
    'Полный адрес помещения',
    'Торговая площадь, вещественный',
    'url'
]).sum()

np.int64(0)

Поскольку дубликатов нет в подмножестве этих признаков, то можем по ним группировать для размножения значений по столбцам "Дата открытия, категориальный" и "Месяц".

In [118]:
categories = ['Новый', 'Средний по возрасту', 'Открыт давно']
df_with_age = df.loc[df.index.repeat(3)].reset_index(drop=True)
df_with_age['Дата открытия, категориальный'] = df_with_age.groupby([
    'Полный адрес помещения',
    'Торговая площадь, вещественный',
    'url'
]).cumcount().apply(lambda i: categories[i])

df_with_month = df_with_age.loc[df_with_age.index.repeat(12)].reset_index(drop=True)
df_with_month['Месяц'] = df_with_month.groupby([
    'Полный адрес помещения',
    'Торговая площадь, вещественный',
    'url'
]).cumcount() + 1

df_with_month.head()

,Полный адрес помещения,"Торговая площадь, вещественный",url,Регион,Населенный пункт,"Торговая площадь, категориальный","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м),"Дата открытия, категориальный",Месяц
0,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Маленький,0,0,2,2,4,0,Новый,1
1,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Маленький,0,0,2,2,4,0,Новый,2
2,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Маленький,0,0,2,2,4,0,Новый,3
3,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Маленький,0,0,2,2,4,0,Новый,4
4,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Маленький,0,0,2,2,4,0,Новый,5


In [119]:
df_with_month.shape

(11844, 14)

In [120]:
train_df = pd.read_excel('../Х5.xlsx')
train_df.head()

,new_id,Месяц,Трафик,Средний чек,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,Регион,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м)
0,0,10,59662,823.060390,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.9,200.333333,0,6,0,0,2,0
1,0,5,56674,859.361975,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.9,200.333333,0,6,0,0,2,0
2,0,1,51488,763.937766,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.9,200.333333,0,6,0,0,2,0
3,0,6,56693,836.362309,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.9,200.333333,0,6,0,0,2,0
4,0,7,58128,845.257709,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,76.9,200.333333,0,6,0,0,2,0


In [121]:
train_df['Медиана количества домохозяйств по нас. пункту'] = (
    train_df.groupby('Населенный пункт')['Количество домохозяйств'].transform('median') # .median() не получится тк возвращет 1 число для группы, а transform() и apply() подгоняют под размер агрегированного датафрэйма
)
train_df['Медиана пешего трафика по нас. пункту'] = (
    train_df.groupby('Населенный пункт')['Трафик пеший, в час'].transform('median')
)
train_df['Медиана авто трафика по нас. пункту'] = (
    train_df.groupby('Населенный пункт')['Трафик авто, в час'].transform('median')
)
train_df.head()

,new_id,Месяц,Трафик,Средний чек,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,Регион,Численность населения,Количество домохозяйств,...,"Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м),Медиана количества домохозяйств по нас. пункту,Медиана пешего трафика по нас. пункту,Медиана авто трафика по нас. пункту
0,0,10,59662,823.060390,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,...,200.333333,0,6,0,0,2,0,670.5,74.95,261.041667
1,0,5,56674,859.361975,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,...,200.333333,0,6,0,0,2,0,670.5,74.95,261.041667
2,0,1,51488,763.937766,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,...,200.333333,0,6,0,0,2,0,670.5,74.95,261.041667
3,0,6,56693,836.362309,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,...,200.333333,0,6,0,0,2,0,670.5,74.95,261.041667
4,0,7,58128,845.257709,Средний по возрасту,Средний,Кавказская ст-ца,Краснодарский край,10177,608,...,200.333333,0,6,0,0,2,0,670.5,74.95,261.041667


In [122]:
feature_to_merge = [
    'Населенный пункт',
    'Численность населения',
    'Медиана количества домохозяйств по нас. пункту',
    'Медиана пешего трафика по нас. пункту',
    'Медиана авто трафика по нас. пункту'
]

non_full_duplicates = train_df[feature_to_merge][~train_df[feature_to_merge].duplicated(keep='first')]
non_full_duplicates[non_full_duplicates.duplicated(subset=['Населенный пункт'], keep='last')]

,Населенный пункт,Численность населения,Медиана количества домохозяйств по нас. пункту,Медиана пешего трафика по нас. пункту,Медиана авто трафика по нас. пункту
36,Подольск г,314934,2639.0,119.636364,160.833333
48,Москва г,12506468,6414.0,158.812500,150.416667
60,Ступино г,64412,860.0,84.333333,125.384615
72,Мытищи г,3396,5375.0,131.032258,120.000000
132,Егорьевск г,71686,1948.5,105.255263,77.377778
...,...,...,...,...,...
239797,Переяславка рп,1315,1151.0,149.857143,223.000000
240786,Бугры г.,0,3764.0,30.857143,104.312500
247649,Троицкое с,3793,361.0,75.500000,174.714286
249981,Троицкое с,12964,361.0,75.500000,174.714286


Как можно заметить, у нас есть дубликаты по городам, хотя такого быть не должно, потому при группировке у них должны были посчитаться одинаковые статистики. В данном случае видим, что статистики разные - значит надо удалить дубликаты. Дубликаты будем удалять по численности населения, здесь есть явные объекты с некорректными значениями данного признака. Например, для Москвы есть 12506468 и 56531, очевидно нужно оставить 12506468. Будем оставлять объект с наибольшим значением признака 'Численность населения'.

In [123]:
df_to_merge_sorted = train_df[feature_to_merge].sort_values('Численность населения', ascending=False)

result_test_df = df_with_month.merge(
    df_to_merge_sorted.drop_duplicates(subset=['Населенный пункт'], keep='first'),
    on='Населенный пункт',
    how='left'
)
result_test_df.head()

,Полный адрес помещения,"Торговая площадь, вещественный",url,Регион,Населенный пункт,"Торговая площадь, категориальный","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м),"Дата открытия, категориальный",Месяц,Численность населения,Медиана количества домохозяйств по нас. пункту,Медиана пешего трафика по нас. пункту,Медиана авто трафика по нас. пункту
0,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Маленький,0,0,2,2,4,0,Новый,1,12506468,6414.0,158.8125,150.416667
1,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Маленький,0,0,2,2,4,0,Новый,2,12506468,6414.0,158.8125,150.416667
2,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Маленький,0,0,2,2,4,0,Новый,3,12506468,6414.0,158.8125,150.416667
3,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Маленький,0,0,2,2,4,0,Новый,4,12506468,6414.0,158.8125,150.416667
4,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,Москва г,Москва г,Маленький,0,0,2,2,4,0,Новый,5,12506468,6414.0,158.8125,150.416667


In [124]:
result_test_df.shape

(11844, 18)

In [125]:
result_test_df = result_test_df[[
    'Полный адрес помещения',
    'Торговая площадь, вещественный',
    'url',
    'Месяц',
    'Дата открытия, категориальный',
    'Торговая площадь, категориальный',
    'Населенный пункт',
    'Регион',
    'Численность населения',
    'Медиана количества домохозяйств по нас. пункту',
    'Медиана пешего трафика по нас. пункту',
    'Медиана авто трафика по нас. пункту',
    'Маркетплейсы, доставки, постаматы (100 м)',
    'Медицинские уч. и аптеки (300 м)',
    'Школы (300 м)',
    'Остановки (300 м)',
    'Продуктовые магазины (500 м)',
    'Пятерочки (500 м)'
]]

result_test_df.rename(columns={
    'Медиана количества домохозяйств по нас. пункту': 'Количество домохозяйств',
    'Медиана пешего трафика по нас. пункту': 'Трафик пеший, в час',
    'Медиана авто трафика по нас. пункту': 'Трафик авто, в час'
}, inplace=True)

result_test_df.head()

,Полный адрес помещения,"Торговая площадь, вещественный",url,Месяц,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,Регион,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м)
0,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,1,Новый,Маленький,Москва г,Москва г,12506468,6414.0,158.8125,150.416667,0,0,2,2,4,0
1,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,2,Новый,Маленький,Москва г,Москва г,12506468,6414.0,158.8125,150.416667,0,0,2,2,4,0
2,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,3,Новый,Маленький,Москва г,Москва г,12506468,6414.0,158.8125,150.416667,0,0,2,2,4,0
3,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,4,Новый,Маленький,Москва г,Москва г,12506468,6414.0,158.8125,150.416667,0,0,2,2,4,0
4,"Москва, ЦАО, р-н Пресненский, м. Краснопреснен...",265.5,https://www.cian.ru/rent/commercial/306607073/...,5,Новый,Маленький,Москва г,Москва г,12506468,6414.0,158.8125,150.416667,0,0,2,2,4,0


In [126]:
if not os.path.exists('../data'):
    os.makedirs('../data')
result_test_df.to_csv('../data/result_test_df.csv', index=False, encoding='utf-8')